In [1]:

from hloc.utils.io import get_matches, get_keypoints
from pathlib import Path
from my_pkg.tools import sort_key, pixel_to_geo_coordinates, read_pairs
import cv2
import numpy as np
loc_path = "/home/lty/outputs/hloc_scene_match_11-21_playground/loc (copy).txt"
output_dir = Path("/home/lty/outputs/hloc_scene_match_11-21_playground")
pairs_path = output_dir/"pairs.txt"
features_path = output_dir/"features.h5"
matches_path = output_dir/"matches.h5"



In [2]:
from my_pkg.tools import extract_rotation_angle, rotate_point_z
pairs = read_pairs(pairs_path)
with open(loc_path, 'w') as loc_file:
    i = 0
    for img_uav, img_tif in pairs:
        if i == 2:
            print(f"UAV: {img_uav} - TIF: {img_tif}")
            kp1, kp2  = get_keypoints(features_path, img_uav), get_keypoints(features_path, img_tif)
            matches,scores = get_matches(matches_path, img_uav, img_tif)
            print(matches.shape)
            pts1 = kp1[matches[:,0]]
            pts2 = kp2[matches[:,1]]
            H, _ = cv2.findHomography(pts1, pts2, cv2.RANSAC)
            #rotation from H
            theta = np.arctan2(H[1,0], H[0,0])
            print(f"theta: {theta}")
            print(H)
            break
        i+=1

# 示例 H 矩阵（请替换为你的实际 H 矩阵）
# H = np.array([
#     [1.2, 0.3, 100],
#     [0.1, 1.1, 50],
#     [0.001, 0.002, 1]
# ])

angle= extract_rotation_angle(H)
print(f"旋转角度 (度): {angle:.2f}")

point1 = np.array([0.076516, 1.667245, -0.17473])
rotated_point1,R= rotate_point_z(point1, angle)
print(R)
print(f"旋转前的点: {point1}")
print(f"旋转后的点: {rotated_point1}")

UAV: seu_uav/00069.png - TIF: seu_tif/2_2000_0.tif
(656, 2)
theta: 0.08147015418558133
[[ 4.31526429e+00 -1.24960879e-01 -1.49499655e+03]
 [ 3.52345142e-01  3.77761637e+00 -1.60979430e+03]
 [ 2.70431353e-04  1.15313206e-05  1.00000000e+00]]
旋转角度 (度): 3.38
[[ 0.99826529 -0.0588762   0.        ]
 [ 0.0588762   0.99826529  0.        ]
 [ 0.          0.          1.        ]]
旋转前的点: [ 0.076516  1.667245 -0.17473 ]
旋转后的点: [-0.02177778  1.66885779 -0.17473   ]
